# Training the NumPy Seq2Seq + Bahdanau Attention Summarizer

This notebook is just the **training** part of the project pulled out of
`train.py` so it can be run cell-by-cell (locally in Jupyter/VS Code, or in
Google Colab).

**Requirements to run this:**
- Must run from inside the `numpy_seq2seq/` folder (or adjust the path setup
  cell below), since it imports `data.py`, `model.py`, `optim.py` from here.
- Needs the `archive/BBC News Summary/` dataset two directories up (i.e. the
  whole project folder, not just this notebook, needs to be present/uploaded).
- Only dependency for the model itself is NumPy. `matplotlib` is only used
  below to plot the loss curve.

Note: this is pure NumPy, so it does **not** use a GPU — running this in
Colab with a GPU runtime gives no speedup. It's CPU-bound either way.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # assumes this notebook runs from numpy_seq2seq/

import time
import numpy as np

from data import prepare_dataset
from model import Seq2SeqAttention
from optim import Adam, clip_grads_

In [ ]:
# Hyperparameters (same defaults as train.py --help)
VOCAB_SIZE = 8000
ENC_MAX_LEN = 60
DEC_MAX_LEN = 20
EMB_DIM = 96
HIDDEN_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
CLIP_NORM = 5.0
SEED = 0
CHECKPOINT_PATH = "checkpoint.npz"

## 1. Load and preprocess the dataset

First run tokenizes the raw text files and caches the result to `data_cache.npz` (fast on later runs).

In [ ]:
ds = prepare_dataset(vocab_size=VOCAB_SIZE, enc_max_len=ENC_MAX_LEN, dec_max_len=DEC_MAX_LEN)
print(f"train examples: {len(ds['enc_ids_train'])}, val examples: {len(ds['enc_ids_val'])}")
print(f"vocab size: {len(ds['itos'])}")

## 2. Build the model and optimizer

In [ ]:
model = Seq2SeqAttention(vocab_size=len(ds["itos"]), emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE, seed=SEED)
optimizer = Adam(model.params, lr=LR)
rng = np.random.RandomState(SEED)

In [ ]:
def iterate_batches(enc_ids, dec_ids, batch_size, rng, shuffle=True):
    n = enc_ids.shape[0]
    order = rng.permutation(n) if shuffle else np.arange(n)
    for start in range(0, n, batch_size):
        idx = order[start:start + batch_size]
        yield enc_ids[idx], dec_ids[idx]

## 3. Training loop

Each batch: forward pass (encoder -> attention -> decoder) -> manual backward pass (BPTT) -> gradient clipping -> Adam step. Saves a checkpoint after every epoch.

In [ ]:
enc_ids_train, dec_ids_train = ds["enc_ids_train"], ds["dec_ids_train"]
num_batches = int(np.ceil(len(enc_ids_train) / BATCH_SIZE))

loss_history = []
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    total_loss_tokens = 0.0
    total_tokens = 0.0
    for enc_batch, dec_batch in iterate_batches(enc_ids_train, dec_ids_train, BATCH_SIZE, rng):
        avg_loss, num_real, cache = model.forward(enc_batch, dec_batch)
        grads = model.backward(cache)
        clip_grads_(grads, max_norm=CLIP_NORM)
        optimizer.step(model.params, grads)

        total_loss_tokens += avg_loss * num_real
        total_tokens += num_real

    epoch_loss = total_loss_tokens / total_tokens
    ppl = float(np.exp(min(epoch_loss, 20)))
    loss_history.append(epoch_loss)
    elapsed = time.time() - epoch_start
    print(f"epoch {epoch}/{EPOCHS} - loss={epoch_loss:.4f} ppl={ppl:.2f} ({elapsed:.1f}s)")

    model.save(CHECKPOINT_PATH)

print(f"Saved final model to {CHECKPOINT_PATH}")

## 4. Plot the training loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.xlabel("epoch")
plt.ylabel("avg loss (cross-entropy per token)")
plt.title("Training loss")
plt.grid(True, alpha=0.3)
plt.show()

## 5. Quick sanity check: summarize a few validation articles

In [ ]:
def ids_to_text(ids, itos):
    words = []
    for i in ids:
        tok = itos[i]
        if tok == "<eos>":
            break
        if tok in ("<pad>", "<sos>"):
            continue
        words.append(tok)
    return " ".join(words)

sample_enc = ds["enc_ids_val"][:3]
sample_ref = ds["dec_ids_val"][:3]
pred_ids, _ = model.greedy_decode(sample_enc, max_len=DEC_MAX_LEN)
for i in range(3):
    print("reference :", ids_to_text(sample_ref[i], ds["itos"]))
    print("prediction:", ids_to_text(pred_ids[i], ds["itos"]))
    print()